In [ ]:
# Colab setup - run this first, then switch the runtime to R
# (Runtime > Change runtime type > R). Precompiled Linux binaries via Posit P3M
# keep the install to a couple of minutes instead of compiling from source.
local({
  cn <- tryCatch(system("lsb_release -cs", intern = TRUE), error = function(e) "jammy")
  options(repos = c(CRAN = sprintf("https://packagemanager.posit.co/cran/__linux__/%s/latest", cn)),
          HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(),
            paste(getRversion(), R.version$platform, R.version$arch, R.version$os)))
})
if (!requireNamespace("BiocManager", quietly = TRUE)) install.packages("BiocManager")
BiocManager::install(
  c("SingleCellExperiment", "SummarizedExperiment", "MultiAssayExperiment",
    "S4Vectors", "basilisk"),
  update = FALSE, ask = FALSE)
install.packages(c("remotes", "reticulate", "Matrix", "ggplot2"))
remotes::install_github("PYangLab/M3", subdir = "m3-r")
# The first m3_train() builds the bundled Python engine via basilisk (a few
# minutes on first run); afterwards it is cached for the session.

# Donor-level disease prediction

m3 can predict a **donor's** disease status from their cells, in a leave-one-batch-out setting: we hold out one batch (`B3`), hide its donors' disease labels during training, and predict them at the end. `m3_train(...)` fits the integration model and the donor predictor in one call; `m3_predict_donors()` then returns per-donor class probabilities.

## 1. Load the demo dataset

In [ ]:
library(m3)
library(ggplot2)
set.seed(0)
data <- m3_demo()
data

## 2. Build + train the model with a held-out batch

On top of the columns from the representation-learning tutorial, donor prediction needs a few more:

- `target_condition`, the condition to predict (here `cond_group`, i.e. disease).
- `donor_key`, the column identifying each donor, so cells are grouped per donor.
- `held_out`, the batch(es) to hold back; these donors' `target_condition` labels are hidden during training and predicted at the end.

The `donor_predictor` list holds the predictor's training settings (learning rate, epochs, and how strongly / on what schedule the batch effect is removed). `m3_reference_vocab(model)` shows the label set the model learned to predict (the held-out donors' labels never enter it).

In [ ]:
#To save time, users can set max_epochs to 100 for test.
model <- m3_train(
  data,
  condition_keys   = c("cond_group", "Age_interval"),
  target_condition = "cond_group",
  celltype_key     = "mergedcelltype",
  batch_key        = "batch",
  donor_key        = "sample_id",
  held_out         = "B3",
  embedding_dim    = 30L,
  max_epochs       = 500L,
  donor_predictor  = list(glr = 3e-3, n_epochs = 120L, adv_max = 10L,
                          adv_warmup = 7L, n_disc = 21L, patient_w = 10L),
  seed = 0L
)
cat("reference vocab (query labels never enter it):",
    paste(m3_reference_vocab(model)$cond_group, collapse = ", "), "\n")

## 3. Predict the held-out donors

`m3_predict_donors(model)` returns one row per held-out donor: the predicted label and the probability of each class.

In [ ]:
preds <- m3_predict_donors(model)
cat("query donors:", nrow(preds), "\n")

print(preds)

## 4. Evaluate against the held-out truth

The held-out donors' real `cond_group` lives in the metadata (never shown to the model). We join it back to score accuracy.

In [ ]:
obs <- m3_dataset_obs(data)
truth <- unique(obs[, c("sample_id", "cond_group")])
truth <- stats::setNames(as.character(truth$cond_group), as.character(truth$sample_id))
preds$true_label <- truth[as.character(preds$sample_id)]
acc <- mean(preds$predicted_label == preds$true_label)
cat(sprintf("held-out accuracy = %.3f\n", acc))

## 5. Patient-level embedding

`m3_donor_embedding()` returns one vector per donor, the donor-level representation the model actually classifies. We UMAP it and colour by reference/query, true label, and whether the held-out prediction was correct.

In [ ]:
demb <- m3_donor_embedding(model)
info <- m3_predict_donors(model, include_reference = TRUE)
info <- info[match(demb$sample_id, info$sample_id), ]
info$true_label <- truth[as.character(demb$sample_id)]
info$set     <- ifelse(info$is_reference, "reference", "query")
info$correct <- ifelse(info$is_reference, "reference",
                       ifelse(info$predicted_label == info$true_label, "correct", "wrong"))

X  <- as.matrix(demb[, grep("^m3_", colnames(demb))])
xy <- m3_umap(X, method = "umap", n_neighbors = 15L, random_state = 0L,
              device = model$device)
pdf <- data.frame(UMAP1 = xy[, 1], UMAP2 = xy[, 2],
                  set = factor(info$set), true_label = factor(info$true_label),
                  correct = factor(info$correct))

pl <- function(colour, title) {
  ggplot(pdf, aes(UMAP1, UMAP2, colour = .data[[colour]])) +
    geom_point(size = 3, alpha = 0.85) + theme_classic() +
    theme(axis.text = element_blank(), axis.ticks = element_blank()) +
    labs(title = title, colour = NULL)
}
print(pl("set",        "Patient embedding — set"))

In [ ]:
print(pl("true_label", "Patient embedding — true_label"))

In [ ]:
print(pl("correct",    "Patient embedding — correct"))

**Done.** Leave-one-batch-out donor prediction, evaluated against the held-out truth, with a patient-level embedding.